# VeriBench Harbor v1.3 Tutorial

This notebook verifies the public `veribench@1.3` release metadata and provides a path for running Harbor oracle smokes.

The full VeriBench is 896 tasks (614 core + 282 IndustrySet). The Hugging Face copy (`brando/veribench`) has 895 rows: one kafka task is held out until its license header is confirmed.

In [ ]:
from __future__ import annotations
import json, os, pathlib, platform, shutil, subprocess, sys, urllib.request

# Configuration
REGISTRY_URL = "https://raw.githubusercontent.com/brando90/harbor-datasets/main/harbor_registry.json"
DATASET_NAME = "veribench"
DATASET_VERSION = "1.3"
DATASET = f"{DATASET_NAME}@{DATASET_VERSION}"
EXPECTED_TASKS = 896
SMOKE_TASK = "aerospace_set_real__my_fprime_crc_payload_size"

def fetch_registry(url: str) -> list[dict]:
    with urllib.request.urlopen(url, timeout=30) as response:
        return json.load(response)

def find_dataset(registry: list[dict], name: str, version: str) -> dict:
    matches = [item for item in registry if item.get("name") == name and str(item.get("version")) == version]
    if len(matches) != 1:
        raise AssertionError(f"Expected exactly one {name}@{version}; found {len(matches)}")
    return matches[0]

# Execution
registry = fetch_registry(REGISTRY_URL)
release = find_dataset(registry, DATASET_NAME, DATASET_VERSION)
tasks = release.get("tasks", [])
metrics = release.get("metrics", [])

print(f"Python {sys.version.split()[0]} on {platform.platform()}")
print(f"OK: found {DATASET} with {len(tasks)} tasks")
print(f"Description: {release.get('description', '')}")
print(f"Metrics: {metrics}")

## Check Runtime Capabilities
Harbor requires Docker for actual execution. This cell checks if Docker and the Harbor CLI are available.

In [ ]:
def command_exists(name: str) -> bool:
    return shutil.which(name) is not None

os.environ["PATH"] += os.pathsep + str(pathlib.Path.home() / ".local" / "bin")

docker_cli = command_exists("docker")
harbor_cli = command_exists("harbor")

print(f"Docker CLI: {docker_cli}")
print(f"Harbor CLI: {harbor_cli}")

## Sample Task Inspection
Let's look at the first few tasks in the dataset.

In [ ]:
print("First 5 task names:")
for task in tasks[:5]:
    print(f"- {task['name']}")

### Summary and Links
- **Harbor Hub**: [hub.harborframework.com](https://hub.harborframework.com/)
- **Harbor Docs**: [harborframework.com/docs](https://harborframework.com/docs/getting-started)
- **Registry Source**: [harbor_registry.json](https://raw.githubusercontent.com/brando90/harbor-datasets/main/harbor_registry.json)